--- Day 10: Pipe Maze ---
You use the hang glider to ride the hot air from Desert Island all the way up to the floating metal island. This island is surprisingly cold and there definitely aren't any thermals to glide on, so you leave your hang glider behind.

You wander around for a while, but you don't find any people or animals. However, you do occasionally find signposts labeled "Hot Springs" pointing in a seemingly consistent direction; maybe you can find someone at the hot springs and ask them where the desert-machine parts are made.

The landscape here is alien; even the flowers and trees are made of metal. As you stop to admire some metal grass, you notice something metallic scurry away in your peripheral vision and jump into a big pipe! It didn't look like any animal you've ever seen; if you want a better look, you'll need to get ahead of it.

Scanning the area, you discover that the entire field you're standing on is densely packed with pipes; it was hard to tell at first because they're the same metallic silver color as the "ground". You make a quick sketch of all of the surface pipes you can see (your puzzle input).

The pipes are arranged in a two-dimensional grid of tiles:

| is a vertical pipe connecting north and south.
- is a horizontal pipe connecting east and west.
L is a 90-degree bend connecting north and east.
J is a 90-degree bend connecting north and west.
7 is a 90-degree bend connecting south and west.
F is a 90-degree bend connecting south and east.
. is ground; there is no pipe in this tile.
S is the starting position of the animal; there is a pipe on this tile, but your sketch doesn't show what shape the pipe has.
Based on the acoustics of the animal's scurrying, you're confident the pipe that contains the animal is one large, continuous loop.

For example, here is a square loop of pipe:

.....
.F-7.
.|.|.
.L-J.
.....
If the animal had entered this loop in the northwest corner, the sketch would instead look like this:

.....
.S-7.
.|.|.
.L-J.
.....
In the above diagram, the S tile is still a 90-degree F bend: you can tell because of how the adjacent pipes connect to it.

Unfortunately, there are also many pipes that aren't connected to the loop! This sketch shows the same loop as above:

-L|F7
7S-7|
L|7||
-L-J|
L|-JF
In the above diagram, you can still figure out which pipes form the main loop: they're the ones connected to S, pipes those pipes connect to, pipes those pipes connect to, and so on. Every pipe in the main loop connects to its two neighbors (including S, which will have exactly two pipes connecting to it, and which is assumed to connect back to those two pipes).

Here is a sketch that contains a slightly more complex main loop:

..F7.
.FJ|.
SJ.L7
|F--J
LJ...
Here's the same example sketch with the extra, non-main-loop pipe tiles also shown:

7-F7-
.FJ|7
SJLL7
|F--J
LJ.LJ
If you want to get out ahead of the animal, you should find the tile in the loop that is farthest from the starting position. Because the animal is in the pipe, it doesn't make sense to measure this by direct distance. Instead, you need to find the tile that would take the longest number of steps along the loop to reach from the starting point - regardless of which way around the loop the animal went.

In the first example with the square loop:

.....
.S-7.
.|.|.
.L-J.
.....
You can count the distance each tile in the loop is from the starting point like this:

.....
.012.
.1.3.
.234.
.....
In this example, the farthest point from the start is 4 steps away.

Here's the more complex loop again:

..F7.
.FJ|.
SJ.L7
|F--J
LJ...
Here are the distances for each tile on that loop:

..45.
.236.
01.78
14567
23...
Find the single giant loop starting at S. How many steps along the loop does it take to get from the starting position to the point farthest from the starting position?

In [156]:
with open('input10.txt') as file:
    data = [l.rstrip() for l in file]
len(data), len(data[-1])

(140, 140)

In [157]:
pipes = {}
"""
| is a vertical pipe connecting north and south.
- is a horizontal pipe connecting east and west.
L is a 90-degree bend connecting north and east.
J is a 90-degree bend connecting north and west.
7 is a 90-degree bend connecting south and west.
F is a 90-degree bend connecting south and east."""
for i, line in enumerate(data):
    for j, p in enumerate(line):
        if p == '|':
            pipes[(i, j)] = ((i-1,j), (i+1,j))
        elif p == '-':
            pipes[(i, j)] = ((i,j-1), (i,j+1))
        elif p == 'L':
            pipes[(i, j)] = ((i-1,j), (i,j+1))   
        elif p == 'J':
            pipes[(i, j)] = ((i-1,j), (i,j-1))   
        # Looking at the puzzle input, only the tiles south and west of the starting tile S connect to it. So we can treat S like a 7 tile:
        elif p == '7' or p == 'S':
            pipes[(i, j)] = ((i+1,j), (i,j-1))
        elif p == 'F':
            pipes[(i, j)] = ((i+1,j), (i,j+1))
        if p =='S':
            start = (i,j)  
start, pipes[start]

((83, 25), ((84, 25), (83, 24)))

In [158]:
loop_len = 0
last = start
current = pipes[start][0]
while current != start:
    loop_len += 1
    new = pipes[current][0] if pipes[current][0] != last else pipes[current][1]
    last = current
    current = new
loop_len

13507

In [159]:
max_dist = loop_len // 2 + loop_len % 2
max_dist

6754

Your puzzle answer was 6754.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
You quickly reach the farthest point of the loop, but the animal never emerges. Maybe its nest is within the area enclosed by the loop?

To determine whether it's even worth taking the time to search for such a nest, you should calculate how many tiles are contained within the loop. For example:

...........
.S-------7.
.|F-----7|.
.||.....||.
.||.....||.
.|L-7.F-J|.
.|..|.|..|.
.L--J.L--J.
...........
The above loop encloses merely four tiles - the two pairs of . in the southwest and southeast (marked I below). The middle . tiles (marked O below) are not in the loop. Here is the same loop again with those regions marked:

...........
.S-------7.
.|F-----7|.
.||OOOOO||.
.||OOOOO||.
.|L-7OF-J|.
.|II|O|II|.
.L--JOL--J.
.....O.....
In fact, there doesn't even need to be a full tile path to the outside for tiles to count as outside the loop - squeezing between pipes is also allowed! Here, I is still within the loop and O is still outside the loop:

..........
.S------7.
.|F----7|.
.||OOOO||.
.||OOOO||.
.|L-7F-J|.
.|II||II|.
.L--JL--J.
..........
In both of the above examples, 4 tiles are enclosed by the loop.

Here's a larger example:

.F----7F7F7F7F-7....
.|F--7||||||||FJ....
.||.FJ||||||||L7....
FJL7L7LJLJ||LJ.L-7..
L--J.L7...LJS7F-7L7.
....F-J..F7FJ|L7L7L7
....L7.F7||L7|.L7L7|
.....|FJLJ|FJ|F7|.LJ
....FJL-7.||.||||...
....L---J.LJ.LJLJ...
The above sketch has many random bits of ground, some of which are in the loop (I) and some of which are outside it (O):

OF----7F7F7F7F-7OOOO
O|F--7||||||||FJOOOO
O||OFJ||||||||L7OOOO
FJL7L7LJLJ||LJIL-7OO
L--JOL7IIILJS7F-7L7O
OOOOF-JIIF7FJ|L7L7L7
OOOOL7IF7||L7|IL7L7|
OOOOO|FJLJ|FJ|F7|OLJ
OOOOFJL-7O||O||||OOO
OOOOL---JOLJOLJLJOOO
In this larger example, 8 tiles are enclosed by the loop.

Any tile that isn't part of the main loop can count as being enclosed by the loop. Here's another example with many bits of junk pipe lying around that aren't connected to the main loop at all:

FF7FSF7F7F7F7F7F---7
L|LJ||||||||||||F--J
FL-7LJLJ||||||LJL-77
F--JF--7||LJLJ7F7FJ-
L---JF-JLJ.||-FJLJJ7
|F|F-JF---7F7-L7L|7|
|FFJF7L7F-JF7|JL---7
7-L-JL7||F7|L7F-7F7|
L.L7LFJ|||||FJL7||LJ
L7JLJL-JLJLJL--JLJ.L
Here are just the tiles that are enclosed by the loop marked with I:

FF7FSF7F7F7F7F7F---7
L|LJ||||||||||||F--J
FL-7LJLJ||||||LJL-77
F--JF--7||LJLJIF7FJ-
L---JF-JLJIIIIFJLJJ7
|F|F-JF---7IIIL7L|7|
|FFJF7L7F-JF7IIL---7
7-L-JL7||F7|L7F-7F7|
L.L7LFJ|||||FJL7||LJ
L7JLJL-JLJLJL--JLJ.L
In this last example, 10 tiles are enclosed by the loop.

Figure out whether you have time to search for the nest by calculating the area within the loop. How many tiles are enclosed by the loop?



In [160]:
# Ok this seems a bit more involved.
# First let's make sure we collect all the locations and symbols of the main loop segments:

main_loop = {start : '7'}
last = start
current = pipes[start][0]

while current != start:
    main_loop[current] = data[current[0]][current[1]]
    new = pipes[current][0] if pipes[current][0] != last else pipes[current][1]
    last = current
    current = new
len(main_loop), main_loop[last], 140**2

(13508, 'F', 19600)

In [161]:
def pipe_pass(heading, blocked_tile, main_loop):
    """return new intersection points (the point between 4 tiles) reachable via blocked_tile travelling from heading. Intersection points are identified by the coordinates of the tile to the bottom right (thus (0,0) is the point at the top left of the grid, while (140,140) is the point at the bottom right)"""
    i, j = blocked_tile
    if heading == 0: # north
        if main_loop[blocked_tile] == 'L':
            return (i, j)
        elif main_loop[blocked_tile] == 'J':
            return (i, j+1)
    elif heading == 1: # south
        if main_loop[blocked_tile] == 'F':
            return (i+1, j)
        elif main_loop[blocked_tile] == '7':
            return (i+1, j+1)
    elif heading == 2: # west
        if main_loop[blocked_tile] == 'J':
            return (i+1, j)
        elif main_loop[blocked_tile] == '7':
            return (i, j)
    elif heading == 3: # east
        if main_loop[blocked_tile] == 'L':
            return (i+1, j+1)
        elif main_loop[blocked_tile] == 'F':
            return (i, j+1)
    return None

In [166]:
def flood_fill(tile, main_loop, in_tiles, out_tiles):
    tile_q = [tile]
    pass_q = []
    tile_v = set()
    pass_v = set()

    while tile_q or pass_q:

        if tile_q:
            new = i, j = tile_q.pop()
            if new not in tile_v:
                tile_v.add(new)
                if i <= 0 or j <= 0 or i >= len(data) - 1 or j >= len(data[0]) - 1 or new in out_tiles:
                    return (tile_v.union(tile_q), "out")
                elif new in in_tiles:
                    return (tile_v.union(tile_q), "in")
                else:
                    neighbors = (i-1, j), (i+1, j), (i, j-1), (i, j+1)
                    for ix, n in enumerate(neighbors):
                        if n in main_loop:
                            if x := pipe_pass(ix, n, main_loop):
                                pass_q.append(x)
                        else:
                            tile_q.append(n)
        
        else: # pass_q
            p = i,j = pass_q.pop()
            if p not in pass_v:
                pass_v.add(p)
                if i <= 0 or j <= 0 or i >= len(data) or j >= len(data[0]):
                    return (tile_v.union(tile_q), "out")          

                neighbors = tl, tr, bl, br = (i-1, j-1), (i-1, j), (i, j-1), (i,j) # adjacent full tiles
                # advance pipe passes (only where no full tile access):
                if main_loop.get(tl, 'x') in '|7J' and main_loop.get(tr, 'x') in '|LF':
                    pass_q.append((i-1, j))
                if main_loop.get(bl, 'x') in '|7J' and main_loop.get(br, 'x') in '|LF':
                    pass_q.append((i+1, j))
                if main_loop.get(tl, 'x') in '-LJ' and main_loop.get(bl, 'x') in '-F7':
                    pass_q.append((i, j-1))
                if main_loop.get(tr, 'x') in '-LJ' and main_loop.get(br, 'x') in '-F7':
                    pass_q.append((i, j+1))

                for n in neighbors:
                    if n not in main_loop:
                        tile_q.append(n)
    # queue emptied:
    return (tile_v, 'in')

In [163]:
in_tiles = set()
out_tiles = set()
for i in range(len(data)):
    for j in range(len(data[0])):
        if (i, j) not in in_tiles and (i, j) not in out_tiles and (i, j) not in main_loop:
            area, status = flood_fill((i, j), main_loop, in_tiles, out_tiles)
            if status == 'in':
                in_tiles.update(area)
            else:
                out_tiles.update(area)
assert len(in_tiles) + len(out_tiles) + len(main_loop) == len(data) * len(data[0])

In [164]:
len(in_tiles), len(out_tiles), len(main_loop)

(567, 5525, 13508)

That's the right answer! You are one gold star closer to restoring snow operations.

You have completed Day 10! You can [Share] this victory or [Return to Your Advent Calendar].